# Hypothesis tests and baseline models

Run after `01_data_preparation_and_eda.ipynb`; saves test results, model splits, and baseline validation metrics.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2_contingency, t, ttest_ind
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "raw_data").is_dir() and (p / "README.md").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

OUTPUT_DIR = PROJECT_ROOT / "processed_data"
SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"
if not SAMPLE_PATH.is_file():
    raise FileNotFoundError(f"Run notebook 01 first; sample not found: {SAMPLE_PATH}")
RANDOM_STATE = 42
sample = pd.read_csv(SAMPLE_PATH)
required = {"Default", "BusinessAge", "Program", "TermInMonths", "GrossApproval", "JobsSupported", "CollateralInd", "NaicsSector", "FranchiseInd"}
missing = required.difference(sample.columns)
if missing:
    raise ValueError(f"Analysis sample is missing required fields: {sorted(missing)}")

for col in ("TermInMonths", "GrossApproval", "JobsSupported"):
    sample[col] = pd.to_numeric(sample[col], errors="coerce").where(lambda values: values >= 0)
sample["LogGrossApproval"] = np.log1p(sample["GrossApproval"])
sample["LogJobsSupported"] = np.log1p(sample["JobsSupported"])
sample["Default"] = sample["Default"].astype(int)
NUMERIC_FEATURES = ["TermInMonths", "LogGrossApproval", "LogJobsSupported"]
CATEGORICAL_FEATURES = ["BusinessAge", "Program", "CollateralInd", "NaicsSector", "FranchiseInd"]
if "RevolverStatus" in sample:
    CATEGORICAL_FEATURES.append("RevolverStatus")
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
print(f"Loaded {len(sample):,} records; default rate={sample['Default'].mean():.3%}.")

In [ ]:
age_table = pd.crosstab(sample["BusinessAge"], sample["Default"]).reindex(columns=[0, 1], fill_value=0)
chi2, p_value, dof, expected = chi2_contingency(age_table)
n = age_table.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * min(age_table.shape[0] - 1, age_table.shape[1] - 1)))
rq1 = pd.DataFrame([{"n": n, "chi2": chi2, "df": dof, "p_value": p_value,
                     "cramers_v": cramers_v, "min_expected_count": expected.min()}])

paid = sample.loc[sample["Default"].eq(0), "TermInMonths"].dropna()
defaulted = sample.loc[sample["Default"].eq(1), "TermInMonths"].dropna()
test = ttest_ind(defaulted, paid, equal_var=False)
mean_difference = defaulted.mean() - paid.mean()
se_difference = np.sqrt(defaulted.var(ddof=1) / len(defaulted) + paid.var(ddof=1) / len(paid))
welch_df = (defaulted.var(ddof=1) / len(defaulted) + paid.var(ddof=1) / len(paid)) ** 2 / (
    (defaulted.var(ddof=1) / len(defaulted)) ** 2 / (len(defaulted) - 1)
    + (paid.var(ddof=1) / len(paid)) ** 2 / (len(paid) - 1)
)
critical = t.ppf(0.975, welch_df)
rq2 = pd.DataFrame([{"default_n": len(defaulted), "paid_n": len(paid),
                     "default_mean_months": defaulted.mean(), "paid_mean_months": paid.mean(),
                     "default_minus_paid_months": mean_difference,
                     "ci95_low": mean_difference - critical * se_difference,
                     "ci95_high": mean_difference + critical * se_difference,
                     "t_statistic": test.statistic, "df": welch_df, "p_value": test.pvalue}])
rq1.to_csv(OUTPUT_DIR / "rq1_business_age_chi_square.csv", index=False)
rq2.to_csv(OUTPUT_DIR / "rq2_term_welch_t_test.csv", index=False)
age_table.to_csv(OUTPUT_DIR / "rq1_business_age_contingency.csv")
display(rq1, rq2)

In [ ]:
train_val, test_set = train_test_split(
    sample, test_size=0.15, random_state=RANDOM_STATE, stratify=sample["Default"]
)
train_set, validation_set = train_test_split(
    train_val, test_size=0.15 / 0.85, random_state=RANDOM_STATE, stratify=train_val["Default"]
)
train_set = train_set.assign(Split="train")
validation_set = validation_set.assign(Split="validation")
test_set = test_set.assign(Split="test")
split_data = pd.concat([train_set, validation_set, test_set], ignore_index=True)
split_data.to_csv(OUTPUT_DIR / "model_data_with_split.csv", index=False)

def make_preprocessor():
    return ColumnTransformer([
        ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_FEATURES),
        ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                  ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL_FEATURES),
    ])

X_train, y_train = train_set[FEATURES], train_set["Default"]
X_validation, y_validation = validation_set[FEATURES], validation_set["Default"]
training_odds = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
baseline_models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "random_forest": RandomForestClassifier(n_estimators=250, min_samples_leaf=5,
                                               class_weight="balanced_subsample", n_jobs=-1,
                                               random_state=RANDOM_STATE),
    "xgboost": XGBClassifier(n_estimators=250, max_depth=5, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8,
                              scale_pos_weight=training_odds, eval_metric="logloss",
                              n_jobs=-1, random_state=RANDOM_STATE),
}

def metrics_at_half(model, X, y):
    probability = model.predict_proba(X)[:, 1]
    prediction = probability >= 0.5
    return {"accuracy": accuracy_score(y, prediction),
            "precision": precision_score(y, prediction, zero_division=0),
            "recall": recall_score(y, prediction, zero_division=0),
            "f1": f1_score(y, prediction, zero_division=0),
            "roc_auc": roc_auc_score(y, probability)}

baseline_results = []
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
for name, estimator in baseline_models.items():
    pipeline = Pipeline([("preprocessor", make_preprocessor()), ("model", estimator)])
    pipeline.fit(X_train, y_train)
    baseline_results.append({"model": name, **metrics_at_half(pipeline, X_validation, y_validation)})
    joblib.dump(pipeline, MODEL_DIR / f"baseline_{name}.joblib")

baseline_results = pd.DataFrame(baseline_results)
baseline_results.to_csv(OUTPUT_DIR / "baseline_validation_metrics.csv", index=False)
display(baseline_results)

In [ ]:
inference_features = NUMERIC_FEATURES + ["BusinessAge", "Program", "CollateralInd", "NaicsSector", "FranchiseInd"]
inference_data = train_set[inference_features + ["Default"]].dropna().copy()
design = pd.get_dummies(inference_data[inference_features], drop_first=True, dtype=float)
design = sm.add_constant(design, has_constant="add")
logit = sm.Logit(inference_data["Default"], design).fit(disp=False, maxiter=200)
confidence = logit.conf_int()
odds_ratios = pd.DataFrame({
    "coefficient": logit.params,
    "odds_ratio": np.exp(logit.params),
    "ci95_low_odds_ratio": np.exp(confidence[0]),
    "ci95_high_odds_ratio": np.exp(confidence[1]),
    "p_value": logit.pvalues,
})
odds_ratios.to_csv(OUTPUT_DIR / "rq3_logistic_regression_odds_ratios.csv")
print(f"RQ3 logistic regression fit on {len(inference_data):,} training records.")
display(odds_ratios.sort_values("p_value").head(20))